
# Time-Series Deep Learning Comparison: LSTM vs RNN vs GRU vs ANN

## Train/Test Split, No-Split Demonstration, Prediction and Evaluation

**Dataset:** Daily Minimum Temperatures in Melbourne, Australia  
**Task:** Predict the next day's temperature from the previous 7 days  
**Framework:** TensorFlow / Keras  
**Level:** Beginner → Intermediate

### What this notebook adds

This notebook extends the original LSTM experiment in two important ways:

1. It compares an LSTM trained/evaluated with a proper chronological train/test split against an LSTM trained on the entire dataset and evaluated on that same data.
2. It builds **LSTM, Simple RNN, GRU and ANN** models as separate experiments and evaluates them on the same test set.

> **Important:** This is a regression problem. The primary metrics are MAE, MSE, RMSE and R², not classification accuracy. A secondary threshold-based accuracy/confusion-matrix section is included only to demonstrate how classification-style evaluation differs from regression evaluation.



# 1. Dataset

The notebook uses the **Daily Minimum Temperatures** dataset.

### Dataset links

- GitHub dataset repository: https://github.com/jbrownlee/Datasets
- Raw CSV: https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv
- Dataset description: https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.names

The CSV contains a `Date` column and a `Temp` column.

### Why this dataset?

It is small enough to run on a normal CPU, but it is useful for understanding:

- time-series sequence creation
- scaling
- chronological train/test splitting
- LSTM
- Simple RNN
- GRU
- ANN baseline
- forecasting metrics
- prediction plots
- overfitting and data leakage



# 2. The Key Question: What Changes With and Without a Train/Test Split?

Suppose we have 3,600 sequence samples.

### Proper train/test experiment

```text
Earlier observations
        ↓
   TRAIN DATA
        ↓
    Model learns
        ↓
Later unseen observations
        ↓
    TEST DATA
        ↓
   Final evaluation
```

The test set represents data the model did not use to update its weights.

### No train/test split

```text
All observations
      ↓
    TRAIN
      ↓
Model learns from all data
      ↓
Evaluate on the same data
```

This does **not** measure generalization to unseen data. The resulting score can look better because the model is being evaluated on data it already trained on.

### Important conclusion

For a real forecasting project, use a **chronological holdout** or another time-series validation strategy. Do not randomly shuffle future and past observations together.

Also note that `train_test_split()` with its default `shuffle=True` is generally not the preferred method for a time-series forecasting problem because it can mix observations from different time periods.


In [ ]:

# Optional installation cell.
# Uncomment if packages are missing.

# %pip install -q tensorflow pandas numpy matplotlib scikit-learn


In [ ]:

import os
import random
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, LSTM, SimpleRNN, GRU
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)



# 3. Download / Load the Dataset

If `daily-min-temperatures.csv` is already in the notebook folder, the first option loads it.

Otherwise, the code downloads the public GitHub copy automatically.


In [ ]:

DATA_URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv"
DATA_FILE = "daily-min-temperatures.csv"

if not os.path.exists(DATA_FILE):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)

df = pd.read_csv(DATA_FILE)

print(df.head())
print("\nShape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:")
print(df.isnull().sum())


In [ ]:

df["Date"] = pd.to_datetime(df["Date"])

plt.figure(figsize=(14, 5))
plt.plot(df["Date"], df["Temp"])
plt.xlabel("Date")
plt.ylabel("Minimum Temperature")
plt.title("Daily Minimum Temperature Time Series")
plt.grid(alpha=0.3)
plt.show()

print("Start date:", df["Date"].min())
print("End date:", df["Date"].max())
print("Temperature statistics:")
print(df["Temp"].describe())



# 4. Create 7-Day Sequences

We use the previous **7 days** to predict the next day.

For example:

```text
Day 1  Day 2  Day 3  Day 4  Day 5  Day 6  Day 7
  ↓      ↓      ↓      ↓      ↓      ↓      ↓
[             INPUT X                    ]
                                      ↓
                                  next day = y
```

The resulting shape is:

```text
X = (samples, 7, 1)
y = (samples, 1)
```

The `1` represents one feature: temperature.


In [ ]:

temperature_raw = df["Temp"].values.astype("float32").reshape(-1, 1)

scaler = MinMaxScaler()
temperature_scaled = scaler.fit_transform(temperature_raw)

WINDOW = 7

X = []
y = []

for i in range(WINDOW, len(temperature_scaled)):
    X.append(temperature_scaled[i-WINDOW:i])
    y.append(temperature_scaled[i])

X = np.array(X, dtype="float32")
y = np.array(y, dtype="float32")

print("X shape:", X.shape)
print("y shape:", y.shape)
print("One input sequence:", X[0].flatten())
print("Target:", y[0])



# 5. Why We Need Two Different LSTM Experiments

We will intentionally perform two experiments.

## Experiment A — No Train/Test Split

The model trains on all sequence samples and is evaluated on those same samples.

This is useful **only as a learning demonstration**. It shows how an in-sample score can be misleading.

## Experiment B — Proper Chronological Train/Test Split

The first 80% of sequences are used for training and the final 20% are held out for testing.

This better represents forecasting on future observations.

### Expected effect

The no-split/in-sample result will often look stronger, while the chronological test result may be worse. That difference is not a failure of the split; it is evidence that unseen data is harder than training data.

The exact numerical difference depends on the run, TensorFlow version, random seed and training settings.



# 6. Helper Functions for Metrics

Because temperature prediction is **regression**, we will use:

- **MAE:** average absolute prediction error
- **MSE:** average squared error
- **RMSE:** square root of MSE; expressed in the original temperature unit
- **R²:** proportion of variance explained by the model
- **MAPE:** mean absolute percentage error

For temperature values near zero, MAPE can become unstable, so treat it as a supplementary metric rather than the only metric.


In [ ]:

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)

    denominator = np.maximum(np.abs(y_true), 1e-6)
    mape = np.mean(np.abs((y_true - y_pred) / denominator)) * 100

    return {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2,
        "MAPE_%": mape,
    }


def show_metrics(name, y_true, y_pred):
    metrics = regression_metrics(y_true, y_pred)
    print(f"\n{name}")
    for key, value in metrics.items():
        print(f"{key:8s}: {value:.4f}")
    return metrics



# 7. LSTM — Experiment A: Train Without a Test Split

### What does "without train/test split" mean here?

The model sees **all available sequences during training**.

After training, we predict those same sequences.

This is an **in-sample evaluation**, not a true test evaluation.

It can be useful for understanding model fitting, but it should not be reported as the final production performance.


In [ ]:

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

lstm_no_split = Sequential([
    Input(shape=(WINDOW, 1)),
    LSTM(50),
    Dense(1)
])

lstm_no_split.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

history_lstm_no_split = lstm_no_split.fit(
    X,
    y,
    epochs=30,
    batch_size=16,
    validation_split=0.0,
    verbose=1
)


In [ ]:

pred_scaled_no_split = lstm_no_split.predict(X, verbose=0)

pred_no_split = scaler.inverse_transform(pred_scaled_no_split)
actual_no_split = scaler.inverse_transform(y)

metrics_lstm_no_split = show_metrics(
    "LSTM — In-Sample / No Test Split",
    actual_no_split,
    pred_no_split
)



### Interpretation

If the no-split metrics look very good, do **not** conclude that the model will perform equally well on future data.

The model has optimized its weights using the same observations used for evaluation.

This is the main reason a no-split score can be overly optimistic.



# 8. LSTM — Experiment B: Proper Chronological Train/Test Split

For forecasting, we preserve time order:

```text
Past 80% --------------------> Future 20%
TRAIN                            TEST
```

The model never uses test targets to update its weights.


In [ ]:

split_index = int(len(X) * 0.80)

X_train = X[:split_index]
y_train = y[:split_index]

X_test = X[split_index:]
y_test = y[split_index:]

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

print("\nTraining period ends at sequence index:", split_index - 1)
print("Testing starts at sequence index:", split_index)



## Important preprocessing note

For a strict production workflow, the scaler should also be fitted using the training period only and then applied to validation/test data.

In this notebook, the initial scaling is kept simple so the main lesson remains visible. Later, you can upgrade the preprocessing pipeline to fit the scaler only on training observations.


In [ ]:

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

lstm_split = Sequential([
    Input(shape=(WINDOW, 1)),
    LSTM(50),
    Dense(1)
])

lstm_split.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

history_lstm_split = lstm_split.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=16,
    validation_split=0.20,
    shuffle=False,
    verbose=1
)


In [ ]:

pred_scaled_lstm = lstm_split.predict(X_test, verbose=0)

pred_lstm = scaler.inverse_transform(pred_scaled_lstm)
actual_lstm = scaler.inverse_transform(y_test)

metrics_lstm_split = show_metrics(
    "LSTM — Chronological Test Set",
    actual_lstm,
    pred_lstm
)


In [ ]:

plt.figure(figsize=(14, 5))
plt.plot(actual_lstm, label="Actual")
plt.plot(pred_lstm, label="Predicted")
plt.title("LSTM — Actual vs Predicted on Unseen Test Data")
plt.xlabel("Test Time Step")
plt.ylabel("Temperature")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



# 9. Compare LSTM: No Split vs Proper Test Set

The table below makes the impact visible.

### What to look for

- Lower MAE is better.
- Lower RMSE is better.
- Lower MSE is better.
- Higher R² is better.
- MAPE is supplementary.

The no-split experiment is an **in-sample** measurement.

The chronological test experiment is an **out-of-sample** measurement.

Therefore, they answer different questions and should not be treated as equivalent evaluation scores.


In [ ]:

lstm_comparison = pd.DataFrame([
    {"Experiment": "LSTM - No Split / In-Sample", **metrics_lstm_no_split},
    {"Experiment": "LSTM - Chronological Test", **metrics_lstm_split},
])

display(lstm_comparison.round(4))



# 10. Why Can Accuracy Change?

For a regression model, there is no natural classification accuracy.

Instead, what changes is the **prediction error**.

A model may show:

```text
No split:
    training data → prediction
    error = relatively small

Chronological test:
    unseen future data → prediction
    error = often larger
```

### Main reasons

1. **Generalization:** unseen observations are different from training observations.
2. **Overfitting:** the model can fit training patterns that do not transfer perfectly.
3. **Temporal distribution changes:** weather patterns can vary over time.
4. **Training data exposure:** no-split evaluation uses observations the model already learned from.
5. **Data leakage:** random splitting can place highly related neighboring time windows in both train and test sets.

### Which approach is better?

For forecasting, the **chronological train/test split is the appropriate evaluation approach** because the goal is to predict future observations.

The no-split experiment is useful for teaching the difference between fitting and generalization, but it should not be used as the final test result.



# 11. LSTM Evaluation: Training Curves

Training and validation loss help diagnose learning behavior.

Typical patterns:

- Training loss decreases and validation loss also decreases → useful learning.
- Training loss keeps decreasing while validation loss rises → possible overfitting.
- Both remain high → possible underfitting or optimization/data problems.


In [ ]:

hist = pd.DataFrame(history_lstm_split.history)

plt.figure(figsize=(8, 4))
plt.plot(hist["loss"], label="Training Loss")
plt.plot(hist["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("LSTM Training vs Validation Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



# 12. Build Four Models on the Same Forecasting Task

We now compare:

1. **ANN** — dense baseline
2. **Simple RNN**
3. **LSTM**
4. **GRU**

All models use the same 7-day input window and the same train/test split.

This makes the comparison more meaningful because the data and target are held constant.



## ANN Architecture

An ANN does not naturally model sequence order.

To use the same 7-day history, we flatten:

```text
(7, 1) → (7,)
```

Then:

```text
7 temperature values
        ↓
Dense
        ↓
Dense
        ↓
Next-day temperature
```

This gives us a useful baseline for comparison.


In [ ]:

def build_ann():
    model = Sequential([
        Input(shape=(WINDOW, 1)),
        layers.Flatten(),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1)
    ])
    model.compile(
        optimizer="adam",
        loss="mean_squared_error"
    )
    return model



## Simple RNN Architecture

A Simple RNN processes the sequence step by step and maintains a hidden state.

```text
Day 1 → Day 2 → Day 3 → ... → Day 7
                  ↓
              RNN state
                  ↓
             next temperature
```

It is simpler than LSTM and GRU, but can struggle with long-term dependencies.


In [ ]:

def build_rnn():
    model = Sequential([
        Input(shape=(WINDOW, 1)),
        SimpleRNN(50),
        Dense(1)
    ])
    model.compile(
        optimizer="adam",
        loss="mean_squared_error"
    )
    return model



## LSTM Architecture

LSTM adds a memory mechanism using gates.

Important ideas:

- forget gate
- input/update gate
- output gate
- cell state
- hidden state

This allows the model to preserve or discard information across time steps.


In [ ]:

def build_lstm():
    model = Sequential([
        Input(shape=(WINDOW, 1)),
        LSTM(50),
        Dense(1)
    ])
    model.compile(
        optimizer="adam",
        loss="mean_squared_error"
    )
    return model



## GRU Architecture

GRU is another gated recurrent architecture.

It generally uses fewer gates than LSTM and can be simpler and faster while still modeling temporal dependencies.

```text
Sequence
   ↓
GRU
   ↓
Dense
   ↓
Next-day temperature
```


In [ ]:

def build_gru():
    model = Sequential([
        Input(shape=(WINDOW, 1)),
        GRU(50),
        Dense(1)
    ])
    model.compile(
        optimizer="adam",
        loss="mean_squared_error"
    )
    return model



# 13. Train ANN, RNN, LSTM and GRU Separately

Each model is trained independently.

The test set is kept untouched until after training.

`shuffle=False` is used because this is a time-series forecasting experiment.


In [ ]:

builders = {
    "ANN": build_ann,
    "RNN": build_rnn,
    "LSTM": build_lstm,
    "GRU": build_gru,
}

trained_models = {}
histories = {}
results = []

for name, builder in builders.items():
    print("\n" + "=" * 60)
    print(f"Training {name}")
    print("=" * 60)

    tf.keras.backend.clear_session()
    tf.random.set_seed(SEED)

    model = builder()

    history = model.fit(
        X_train,
        y_train,
        epochs=30,
        batch_size=16,
        validation_split=0.20,
        shuffle=False,
        verbose=1
    )

    pred_scaled = model.predict(X_test, verbose=0)
    pred = scaler.inverse_transform(pred_scaled)
    actual = scaler.inverse_transform(y_test)

    metrics = regression_metrics(actual, pred)
    results.append({"Model": name, **metrics})

    trained_models[name] = model
    histories[name] = history

    print(f"{name} test metrics:", metrics)



# 14. Model Comparison

This is the main comparison table.

Remember:

- **MAE ↓** better
- **MSE ↓** better
- **RMSE ↓** better
- **R² ↑** better
- **MAPE ↓** generally better, but it is sensitive near zero


In [ ]:

results_df = pd.DataFrame(results)
display(results_df.round(4))


In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(results_df["Model"], results_df["MAE"])
axes[0].set_title("MAE — Lower is Better")
axes[0].set_ylabel("MAE")

axes[1].bar(results_df["Model"], results_df["RMSE"])
axes[1].set_title("RMSE — Lower is Better")
axes[1].set_ylabel("RMSE")

axes[2].bar(results_df["Model"], results_df["R2"])
axes[2].set_title("R² — Higher is Better")
axes[2].set_ylabel("R²")

plt.tight_layout()
plt.show()



# 15. Prediction for Every Model

We now generate test predictions separately for:

- ANN
- RNN
- LSTM
- GRU

The predictions are converted back from the scaled range to the original temperature unit.


In [ ]:

predictions_by_model = {}

actual_original = scaler.inverse_transform(y_test)

for name, model in trained_models.items():
    pred_scaled = model.predict(X_test, verbose=0)
    predictions_by_model[name] = scaler.inverse_transform(pred_scaled).reshape(-1)

prediction_table = pd.DataFrame({
    "Actual": actual_original.reshape(-1)
})

for name, pred in predictions_by_model.items():
    prediction_table[f"{name}_Prediction"] = pred

display(prediction_table.head(20).round(2))



# 16. Actual vs Predicted Plot for All Models

This visualization helps answer:

- Does the prediction follow the trend?
- Does the model lag behind sudden changes?
- Does it over-smooth the signal?
- Which model appears to capture the sequence better?


In [ ]:

plt.figure(figsize=(15, 7))

plt.plot(actual_original.reshape(-1), label="Actual", linewidth=2)

for name, pred in predictions_by_model.items():
    plt.plot(pred, label=name, alpha=0.8)

plt.title("Actual vs Predicted Temperature — Test Set")
plt.xlabel("Test Time Step")
plt.ylabel("Temperature")
plt.legend()
plt.grid(alpha=0.3)
plt.show()



# 17. Residual Analysis

A residual is:

**Residual = Actual − Predicted**

Residual analysis can reveal systematic errors.

Ideally, residuals should not show a strong trend with time or predicted value.


In [ ]:

plt.figure(figsize=(14, 8))

for i, (name, pred) in enumerate(predictions_by_model.items(), start=1):
    residuals = actual_original.reshape(-1) - pred

    plt.subplot(2, 2, i)
    plt.scatter(pred, residuals, alpha=0.25)
    plt.axhline(0, linestyle="--")
    plt.title(f"{name} Residuals")
    plt.xlabel("Predicted Temperature")
    plt.ylabel("Residual")

plt.tight_layout()
plt.show()



# 18. Prediction Error Distribution

The error distribution helps us understand how large prediction errors are and whether the model has extreme misses.


In [ ]:

plt.figure(figsize=(12, 6))

for name, pred in predictions_by_model.items():
    errors = actual_original.reshape(-1) - pred
    plt.hist(errors, bins=40, alpha=0.35, label=name)

plt.axvline(0, linestyle="--")
plt.xlabel("Prediction Error (Actual - Predicted)")
plt.ylabel("Frequency")
plt.title("Prediction Error Distribution")
plt.legend()
plt.show()



# 19. Per-Model Test Evaluation

The following section prints the full regression evaluation for every model.

### Interpretation

- MAE tells you the average absolute temperature error.
- RMSE penalizes large errors more strongly than MAE.
- R² measures how much variance is explained relative to a mean-based baseline.
- MAPE is a percentage-style error but should be interpreted carefully for values near zero.


In [ ]:

for name, pred in predictions_by_model.items():
    show_metrics(
        f"{name} — Test Set",
        actual_original,
        pred
    )



# 20. Is "Accuracy" Valid for This Problem?

**Not as the primary metric.**

Temperature prediction is a continuous regression task.

For example:

```text
Actual temperature = 18.4°C
Predicted temperature = 17.9°C
```

The prediction is not simply "correct" or "incorrect".

Therefore, MAE/RMSE/MSE/R² are more informative.

However, to demonstrate a confusion matrix, we can create a **secondary classification problem**:

> Is tomorrow's temperature above or below the training-set median temperature?

This is only an additional diagnostic. It is not the primary forecasting evaluation.


In [ ]:

threshold = float(np.median(
    scaler.inverse_transform(y_train).reshape(-1)
))

print("Classification threshold:", threshold)

classification_results = []

for name, pred in predictions_by_model.items():
    actual_binary = (actual_original.reshape(-1) >= threshold).astype(int)
    pred_binary = (pred >= threshold).astype(int)

    acc = accuracy_score(actual_binary, pred_binary)
    cm = confusion_matrix(actual_binary, pred_binary)

    classification_results.append({
        "Model": name,
        "Threshold Accuracy": acc
    })

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)
    print("Threshold accuracy:", round(acc, 4))
    print("\nConfusion matrix:")
    print(cm)
    print("\nClassification report:")
    print(classification_report(
        actual_binary,
        pred_binary,
        target_names=["Below threshold", "Above threshold"],
        zero_division=0
    ))



# 21. Confusion Matrices for the Secondary Classification View

A confusion matrix is appropriate when the target is categorical.

Here the categories are:

- `0` = below the training median
- `1` = at/above the training median

Again, this is a **secondary classification interpretation of the regression output**, not a replacement for MAE/RMSE/R².


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(10, 9))

for ax, (name, pred) in zip(axes.ravel(), predictions_by_model.items()):
    actual_binary = (actual_original.reshape(-1) >= threshold).astype(int)
    pred_binary = (pred >= threshold).astype(int)

    cm = confusion_matrix(actual_binary, pred_binary)

    ax.imshow(cm)
    ax.set_title(f"{name} — Threshold Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1], ["Below", "Above"])
    ax.set_yticks([0, 1], ["Below", "Above"])

    for r in range(2):
        for c in range(2):
            ax.text(c, r, cm[r, c], ha="center", va="center")

plt.tight_layout()
plt.show()



# 22. Model Parameter Counts

Parameter count tells you how many trainable weights a model contains.

More parameters do not automatically mean better forecasting.

A larger model can:

- learn more complex patterns
- require more computation
- require more data
- overfit more easily


In [ ]:

parameter_table = []

for name, model in trained_models.items():
    parameter_table.append({
        "Model": name,
        "Parameters": model.count_params()
    })

parameter_df = pd.DataFrame(parameter_table)
display(parameter_df)



# 23. Training Curves for All Models

Compare training and validation loss.

This helps diagnose:

- overfitting
- underfitting
- unstable learning
- differences in convergence speed


In [ ]:

plt.figure(figsize=(14, 9))

for i, (name, history) in enumerate(histories.items(), start=1):
    h = pd.DataFrame(history.history)

    plt.subplot(2, 2, i)
    plt.plot(h["loss"], label="Train")
    plt.plot(h["val_loss"], label="Validation")
    plt.title(f"{name} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.legend()
    plt.grid(alpha=0.2)

plt.tight_layout()
plt.show()



# 24. Forecast One New Next-Day Temperature

We can take the most recent 7 observed temperatures and ask each trained model for the next-day forecast.

```text
Latest 7 days
     ↓
ANN / RNN / LSTM / GRU
     ↓
Next-day temperature prediction
```


In [ ]:

latest_7 = temperature_scaled[-WINDOW:].reshape(1, WINDOW, 1)

next_day_predictions = {}

for name, model in trained_models.items():
    pred_scaled = model.predict(latest_7, verbose=0)
    pred_original = scaler.inverse_transform(pred_scaled)[0, 0]
    next_day_predictions[name] = float(pred_original)

next_day_df = pd.DataFrame(
    list(next_day_predictions.items()),
    columns=["Model", "Next_Day_Prediction"]
)

display(next_day_df.round(2))



# 25. Save the Models

In a real workflow, trained models should be saved so that inference does not require retraining.


In [ ]:

os.makedirs("saved_models", exist_ok=True)

for name, model in trained_models.items():
    file_path = f"saved_models/{name.lower()}_temperature.keras"
    model.save(file_path)
    print("Saved:", file_path)



# 26. Final Comparison — What Should You Report?

For a forecasting project, report the **chronological test-set metrics**.

Do not report the no-split/in-sample score as if it were an unseen test score.

A useful project report should contain:

1. Dataset description
2. Sequence/window definition
3. Train/test strategy
4. Preprocessing/scaling strategy
5. ANN architecture
6. RNN architecture
7. LSTM architecture
8. GRU architecture
9. Parameter counts
10. Training curves
11. MAE
12. MSE
13. RMSE
14. R²
15. Prediction plots
16. Residual analysis
17. Error distribution
18. Optional threshold-based classification diagnostics
19. Final next-day predictions
20. Model saving/inference process



# 27. Key Conceptual Conclusions

### Train/Test Split

A test set contains observations that are not used to update model weights.

### No Train/Test Split

Training and evaluation happen on the same observations. This measures fitting to known data, not generalization.

### Why test performance can be worse

Unseen future observations may contain patterns that are harder to predict. A proper test set exposes this.

### Why chronological split is preferred

Forecasting is directional in time:

```text
Past → Future
```

A model should learn from the past and be evaluated on later observations.

### LSTM vs RNN vs GRU vs ANN

- **ANN:** strong baseline but does not naturally model sequential memory.
- **Simple RNN:** designed for sequences but may have difficulty with longer dependencies.
- **LSTM:** uses gates and a cell state to preserve useful information.
- **GRU:** gated recurrent model with a simpler structure than LSTM.

The numerical results in this notebook should be interpreted from the actual run rather than assuming one architecture always wins.



# 28. Practice Exercises

## Beginner

1. Change the sequence window from 7 to 14.
2. Change epochs from 30 to 50.
3. Change batch size from 16 to 32.
4. Compare LSTM units 32, 50 and 64.
5. Add a second Dense layer to the ANN.

## Intermediate

6. Add Dropout to the ANN.
7. Add Dropout to the LSTM and GRU.
8. Compare Adam and RMSprop.
9. Compare different learning rates.
10. Create a table containing training time and test RMSE.
11. Plot only the last 100 test observations.
12. Compare errors during colder vs warmer periods.

## Advanced

13. Fit the scaler only on the training period.
14. Build a multi-step forecast for the next 7 days.
15. Add multiple features instead of temperature only.
16. Add callbacks such as EarlyStopping.
17. Perform walk-forward validation.
18. Add attention after the LSTM.
19. Export the best model and create a prediction API.
20. Track experiments with MLflow.



# 29. Interview / Viva Questions

1. Why should time-series data normally be split chronologically?
2. What is the difference between training data and test data?
3. Why can evaluating on the training data give an overly optimistic result?
4. Why is accuracy not the primary metric for temperature regression?
5. What is MAE?
6. What is RMSE?
7. Why does RMSE penalize large errors more strongly?
8. What does R² represent?
9. What is the purpose of MinMaxScaler?
10. Why should preprocessing be fitted using training data in a strict production pipeline?
11. What is a sequence/window?
12. What is the difference between RNN, LSTM and GRU?
13. Why can Simple RNN struggle with long-term dependencies?
14. What problem do LSTM gates help address?
15. Why is ANN useful as a baseline?
16. What is overfitting?
17. How can training and validation curves reveal overfitting?
18. Why can random train/test splitting be problematic for forecasting?
19. What does a residual represent?
20. When is a confusion matrix appropriate?



# 30. Final Takeaway

For this forecasting problem:

```text
Raw time series
      ↓
Scaling
      ↓
7-day sequences
      ↓
Chronological train/test split
      ↓
ANN / RNN / LSTM / GRU
      ↓
Train each model separately
      ↓
Predict unseen test data
      ↓
MAE / MSE / RMSE / R² / MAPE
      ↓
Prediction plots + residual analysis
      ↓
Optional threshold accuracy + confusion matrix
```

**The most important lesson is not which model gets the highest number. The important lesson is whether the evaluation setup measures genuine generalization to unseen future data.**
